# Usar o tipo de concessão Authorization code grant no fluxo de oOutbound Auth para Targets do AgentCore Gateway

## Visão Geral
A autenticação/outbound auth permite que os gateways do Amazon Bedrock AgentCore acessem de forma segura os targets do gateway em nome de usuários que foram autenticados e autorizados durante a inbound auth. O AgentCore Gateway suporta múltiplos tipos de outbound auth, como sem autorização, outbound auth baseada em IAM, OAuth e API key.

Você pode usar os seguintes tipos de concessões de autorização OAuth:
* **Client credentials grant** – Autenticação máquina-a-máquina (também conhecida como OAuth de 2 pernas). A aplicação cliente acessa recursos em nome da própria aplicação, e não em nome do usuário.
* **Authorization code grant (Novo)** – Acesso delegado pelo usuário (também conhecido como OAuth de 3 pernas). O usuário fornece consentimento para que a aplicação cliente acesse recursos em seu nome.

Com a **versão 2025-11-25** da especificação do Model Context Protocol (MCP), o MCP suporta [URL Mode Elicitation](https://blog.modelcontextprotocol.io/posts/2025-11-25-first-mcp-anniversary/#url-mode-elicitation-secure-out-of-band-interactions). O URL mode elicitation permite enviar os usuários para um fluxo OAuth adequado (ou qualquer fluxo de aquisição de credenciais) em seu navegador, onde eles podem se autenticar de forma segura sem que seu cliente veja as credenciais inseridas. As credenciais são então gerenciadas diretamente pelo servidor e o cliente só precisa se preocupar com seu próprio fluxo de autorização para o servidor.

O AgentCore gateway agora suporta **Authorization code grant** como Outbound Auth para gateways criados com 2025-11-25 como versão MCP suportada. O Authorization code grant é útil ao acessar ferramentas em nome de um usuário, como Google (e-mail, calendário etc.), Github (repositórios, pull requests etc.), Linkedin (perfil do usuário, publicações etc.) OU ferramentas internas. O fluxo de Authorization code grant prioriza a privacidade do usuário ao não expor as credenciais do usuário a aplicações de terceiros.

Neste tutorial, criaremos um AgentCore Gateway com ferramentas do Linkedin como target usando Authorization code grant. O AgentCore gateway resultante pode então ser usado para construir agentes que podem automatizar ações no Linkedin, como obter informações do usuário, ler/resumir publicações ou até mesmo criar novas publicações etc. em nome do usuário.


## Escolhendo o padrão correto de autenticação/autorização
Ao projetar sua estratégia de autenticação/autorização de agentes, considere estes fatores para determinar qual padrão é mais apropriado:

| **Fator** | **OAuth 2.0 authorization code grant (Acesso delegado pelo usuário)** | **OAuth 2.0 client credentials grant (Autenticação máquina-a-máquina)** |
|------------|----------------------------------------------------------------|-----------------------------------------------------------------------------|
| **Propriedade dos dados** | Dados específicos do usuário (e-mails, documentos, calendários pessoais) | Dados de propriedade do sistema ou organização (analytics, logs, recursos compartilhados) |
| **Interação do usuário** | O usuário está presente e pode fornecer consentimento | Nenhuma interação do usuário é necessária ou disponível |
| **Momento da operação** | Operações interativas, em tempo real | Operações em background, agendadas ou em lote |
| **Escopo de permissões** | As permissões variam por usuário e suas escolhas de consentimento | Permissões consistentes definidas no nível do agente |

## Características principais do authorization code grant

* Requer consentimento explícito do usuário através de um prompt de autorização
* Fornece acesso a dados e recursos específicos do usuário
* Mantém separação clara entre a identidade do agente e a autorização do usuário
* Suporta escopos granulares que limitam quais dados o agente pode acessar

### Arquitetura do Tutorial

<div style="text-align:center">
    <img src="images/outbound_auth_3lo.png" width="90%"/>
</div>


### Detalhes do Tutorial

| Informação          | Detalhes                                                                 |
|:--------------------|:-------------------------------------------------------------------------|
| Tipo de tutorial    | Interativo                                                               |
| Tipo de agente      | MCP Client                                                               |
| Framework de agente | MCP SDK                                                                  |
| Modelo LLM          | N/A                                                                      |
| Componentes do tutorial | AgentCore Gateway com target Linkedin (usando OAuth 2.0 authorization code grant) |
| Vertical do tutorial | Cross-vertical                                                          |
| Complexidade do exemplo | Média                                                                |
| SDK utilizado       | Amazon BedrockAgentCore Python SDK e Boto3                               |
| Credential Provider | Tipo: OAuth2 - Linkedin Provider                                        |


## Pré-requisitos

Para executar este tutorial você precisará de:
* Python 3.10+
* Credenciais AWS
* UV

In [ ]:
# Install from the requirements file in current directory
!uv pip install --system --force-reinstall --no-cache-dir -r requirements.txt --quiet

In [ ]:
# Import required libraries and generate a unique timestamp for naming resources.

import boto3
import json
import time
import zipfile
import io
import os
import sys
import requests
from botocore.auth import SigV4Auth
from botocore.awsrequest import AWSRequest
from botocore.exceptions import ClientError
from datetime import datetime

print("✓ Libraries imported")

# Generate timestamp for unique naming
timestamp = datetime.now().strftime('%Y%m%d-%H%M%S')
print(f"Using timestamp: {timestamp}")

REGION = os.environ['AWS_REGION']

gateway_target_name = f"mcp-target-{timestamp}"

In [ ]:
# Import Utilities and Configure Logging

# Get the directory of the current script
if '__file__' in globals():
    current_dir = os.path.dirname(os.path.abspath(__file__))
else:
    current_dir = os.getcwd()  # Fallback if __file__ is not defined (e.g., Jupyter)

# Navigate to the directory containing utils.py (one level up)
utils_dir = os.path.abspath(os.path.join(current_dir, '..'))

# Add to sys.path
sys.path.insert(0, utils_dir)

# Now import utils
import utils

# Setup logging 
import logging

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(name)s | %(message)s",
    handlers=[logging.StreamHandler()]
)

logging.getLogger("strands").setLevel(logging.INFO)

print("✓ Logging configured, utils imported")

## Configurar Inbound Auth com Amazon Cognito como IDP

Vamos provisionar um Cognito Userpool com um App client. Usaremos o Amazon Cognito para fornecer tokens JWT para acessar nosso servidor MCP implantado.

In [ ]:
USER_POOL_NAME = f"agentcore-gateway-authcode-pool-{timestamp}"
RESOURCE_SERVER_ID = f"agentcore-gateway-authcode-id-{timestamp}"
RESOURCE_SERVER_NAME = f"agentcore-gateway-authcode-name-{timestamp}"
CLIENT_NAME = f"agentcore-gateway-authcode-client-{timestamp}"

# Scopes are based on the current gateway_target_name for THIS run
SCOPES = [
    # Full access to MCP target
    {
        "ScopeName": gateway_target_name,
        "ScopeDescription": "Full access to all tools in MCP target"
    }
]

# Full scope strings in Cognito format: "<resource-server-id>/<scope-name>"
scope_names = [f"{RESOURCE_SERVER_ID}/{scope['ScopeName']}" for scope in SCOPES]
scopeString = " ".join(scope_names)

cognito = boto3.client("cognito-idp", region_name=REGION)

print("Creating or retrieving Cognito resources...")
gw_user_pool_id = utils.get_or_create_user_pool(cognito, USER_POOL_NAME)
print(f"User Pool ID: {gw_user_pool_id}")

utils.get_or_create_resource_server(
    cognito,
    gw_user_pool_id,
    RESOURCE_SERVER_ID,
    RESOURCE_SERVER_NAME,
    SCOPES
)
print("Resource server ensured.")

gw_client_id, gw_client_secret = utils.get_or_create_m2m_client(
    cognito,
    gw_user_pool_id,
    CLIENT_NAME,
    RESOURCE_SERVER_ID,
    scope_names
)
print(f"Client ID: {gw_client_id}")

# Discovery URL used later by the Gateway authorizer and utils.get_token
gw_cognito_discovery_url = (
    f"https://cognito-idp.{REGION}.amazonaws.com/{gw_user_pool_id}/.well-known/openid-configuration"
)
print(gw_cognito_discovery_url)

## Configurar Linkedin como credential provider para OAuth2 de Saída

### Passo 1: Obter o Client Id e Client Secret do LinkedIn

Usaremos as APIs do LinkedIn para testar o fluxo de Auth code grant. Primeiro, você precisará obter o ClientId e ClientSecret do LinkedIn, para que possa obter o access token para acessar as APIs do LinkedIn. Para obtê-los:

* Acesse https://developer.linkedin.com/
* Clique em criar app e adicione um nome para o app
* Você precisará criar uma página do LinkedIn como etapa obrigatória. Quando clicar em For Business, você terá a opção de criar uma página. Crie qualquer página fictícia.
* Após criar um App, você o verá na seção My Apps do portal de desenvolvedores
* Na seção Auth do seu App, você encontrará o clientId e clientSecret que pode usar para obter access tokens.
* Na seção Products do seu App, você precisa habilitar quais APIs são acessíveis. Clique em "Request Access" para "Sign In with LinkedIn using OpenID Connect"
* Isso habilitará a API para obter informações do perfil do usuário.

<div style="text-align:center">
    <img src="images/linkedin-oauth1.png" width="90%"/>
</div>

<div style="text-align:center">
    <img src="images/linkedin-oauth2.png" width="90%"/>
</div>

Nota: Há também uma seção chamada Oauth 2.0 settings, na qual você precisa fornecer Authorized Redirect URLs. Voltaremos a esta seção após criar o credential provider no próximo passo.

### Passo 2: Criar Linkedin Credential provider com AgentCore Identity
O Amazon Bedrock AgentCore Identity fornece providers OAuth 2.0 gerenciados para inbound auth e saída. Cada provider encapsula os protocolos de autenticação específicos, configurações de endpoint e formatos de credenciais necessários para um serviço ou sistema de identidade particular. O serviço fornece providers integrados para serviços populares incluindo Google, GitHub, Slack e Salesforce com endpoints de servidor de autorização e parâmetros específicos do provider pré-configurados para reduzir o esforço de desenvolvimento. Os providers abstraem a complexidade de diferentes implementações OAuth 2.0, esquemas de autenticação de API e formatos de token, apresentando uma interface unificada para agentes enquanto lidam com as variações e casos especiais do protocolo subjacente.

Para este tutorial, criaremos um credential provider usando Linkedin como provider integrado.

**Por favor, insira o Client Id e Client Secret do Linkedin conforme anotado anteriormente.**

In [ ]:
target_client_id = "<clientid>" #replace this with client id from https://developer.linkedin.com/
target_client_secret = "<clientsecret>" #replace this with client secret from https://developer.linkedin.com/

target_cred_provider_name = f"ac-gateway-mcp-server-identity-authcode-{timestamp}"

identity_client = boto3.client('bedrock-agentcore-control', region_name=REGION)

print(f"Deleting the credential provider with name {target_cred_provider_name} if it exists already")
try:
    delete_resp = identity_client.delete_oauth2_credential_provider(name=target_cred_provider_name)
    print("Existing credential provider found and deleted. Proceeding with re-creation")
except ClientError as e:
    if e.response['Error']['Code'] == 'ResourceNotFound':
        print("Existing credential provider with same name does not exist. Proceeding with creation")
    else:
        raise Exception(f"Credential provider deletion failed: {e.response['Error']['Message']}")

linkedin_cred_provider = identity_client.create_oauth2_credential_provider(
    name=target_cred_provider_name,
    credentialProviderVendor="LinkedinOauth2",
    oauth2ProviderConfigInput={
        'linkedinOauth2ProviderConfig': {
            'clientId': target_client_id,
            'clientSecret': target_client_secret
        }
    }
)

target_cred_provider_arn = linkedin_cred_provider['credentialProviderArn']
target_callback_url = linkedin_cred_provider['callbackUrl']
print("Outbound OAuth2 Credential Provider ARN:", target_cred_provider_arn)
print("Please register the following callback URL with linkedin ", target_callback_url)

### Passo 3: Registrar a callback URL no linkedin
Registre a callback URL recebida como saída da célula anterior no App do Linkedin.

<div style="text-align:center">
    <img src="images/linkedin-oauth3.png" width="90%"/>
</div>

## Criar AgentCore gateway com Cognito como inbound auth
Vamos criar um AgentCore Gateway com Cognito como provider de Inbound Auth

In [ ]:
def create_gateway_with_authcode():
    iam_client = boto3.client('iam', region_name=REGION)
    gateway_client = boto3.client('bedrock-agentcore-control', region_name=REGION)
    
    role_name = f'BedrockAgentCoreGatewayRole-{timestamp}'
    gateway_name = f"gateway-authcode-{timestamp}"
    
    # Create IAM role for Gateway
    trust_policy = {
        "Version": "2012-10-17",
        "Statement": [
            {
                "Effect": "Allow",
                "Principal": {
                    "Service": "bedrock-agentcore.amazonaws.com"
                },
                "Action": "sts:AssumeRole"
            }
        ]
    }

    try:
        iam_response = iam_client.create_role(
            RoleName=role_name,
            AssumeRolePolicyDocument=json.dumps(trust_policy),
            Description='IAM role for Bedrock Agent Core Gateway with 3LO'
        )

        role_arn = iam_response['Role']['Arn']
        print(f"Gateway IAM role created: {role_arn}")

        # Attach admin policy to the role
        iam_client.attach_role_policy(
            RoleName=role_name,
            PolicyArn='arn:aws:iam::aws:policy/AdministratorAccess'
        )
        print("Admin policy attached to Gateway IAM role")
    except ClientError as e:
        if e.response['Error']['Code'] == 'EntityAlreadyExists':
            iam_response = iam_client.get_role(RoleName=role_name)
            role_arn = iam_response['Role']['Arn']
            print(f"IAM role already exists with Arn: {role_arn}. Using the same")
        else:
            raise Exception(f"IAM Role creation failed: {e.response['Error']['Message']}")

    print("Creating gateway with Auth Code grant...")
    gateway_response = gateway_client.create_gateway(
        name=gateway_name,
        protocolType="MCP",
        protocolConfiguration={
            "mcp": {
                "supportedVersions": ["2025-03-26", "2025-11-25"],
                "searchType": "SEMANTIC"
            }
        },
       authorizerType="CUSTOM_JWT",
       authorizerConfiguration={
            "customJWTAuthorizer": {
                "discoveryUrl": gw_cognito_discovery_url,
                "allowedClients": [gw_client_id]
            }
       },
       roleArn=role_arn 
    )

    print("Gateway create response:", gateway_response)

    gateway_id = gateway_response['gatewayId']
    gateway_url = gateway_response['gatewayUrl']
    print(f"Gateway with Auth code grant created: {gateway_id}")

    # Wait for gateway to be ready
    print("Waiting for gateway to be ready...")
    while True:
        status_response = gateway_client.get_gateway(gatewayIdentifier=gateway_id)

        current_status = status_response['status']
        print(f"Gateway status: {current_status}")
        if current_status == 'READY':
            print(f"Final gateway details: {status_response}")
            break

        time.sleep(10)

    print("Gateway is now ready")
    return gateway_id, gateway_url, role_name

gateway_id, gateway_url, gateway_role_name = create_gateway_with_authcode()
print(f"\n✅ Gateway creation completed: Gateway Id {gateway_id}")
print(f"Gateway Url: {gateway_url}")

## Criar um target com AgentCore gateway
Usaremos uma especificação OpenAPI do Linkedin contendo a ferramenta para **/userInfo** e criaremos o target com o AgentCore Gateway.

In [ ]:
linkedin_openapi_spec = {
  "openapi": "3.0.0",
  "info": {
    "title": "LinkedIn UserInfo API",
    "version": "2.0.0"
  },
  "servers": [
    {
      "url": "https://api.linkedin.com/v2"
    }
  ],
  "paths": {
    "/userinfo": {
      "get": {
        "operationId": "getUserInfo",
        "summary": "Get User Information",
        "security": [
          {
            "BearerAuth": []
          }
        ],
        "responses": {
          "200": {
            "description": "Successful response",
            "content": {
              "application/json": {
                "schema": {
                  "$ref": "#/components/schemas/UserInfo"
                },
                "example": {
                  "sub": "782bbtaQ",
                  "name": "John Doe",
                  "given_name": "John",
                  "family_name": "Doe",
                  "picture": "https://media.licdn-ei.com/dms/image/C5F03AQHqK8v7tB1HCQ/profile-displayphoto-shrink_100_100/0/",
                  "locale": "en-US",
                  "email": "doe@email.com",
                  "email_verified": True
                }
              }
            }
          }
        }
      }
    }
  },
  "components": {
    "schemas": {
      "UserInfo": {
        "type": "object",
        "properties": {
          "sub": {
            "type": "string"
          },
          "name": {
            "type": "string"
          },
          "given_name": {
            "type": "string"
          },
          "family_name": {
            "type": "string"
          },
          "picture": {
            "type": "string"
          },
          "locale": {
            "type": "string"
          },
          "email": {
            "type": "string"
          },
          "email_verified": {
            "type": "boolean"
          }
        }
      }
    },
    "securitySchemes": {
      "BearerAuth": {
        "type": "http",
        "scheme": "bearer"
      }
    }
  }
}

In [ ]:
DEFAULT_RETURN_URL = "http://localhost:3021" # using some dummy URL as default. We will override when making JSON RPC calls

def create_linkedin_target(gatewayId, provider_arn):
    gateway_client = boto3.client('bedrock-agentcore-control', region_name=REGION)
    credentialProviderConfig = {
        "credentialProviderType": "OAUTH",
        "credentialProvider": {
            "oauthCredentialProvider": {
                "providerArn": provider_arn,
                "grantType": "AUTHORIZATION_CODE",
                "defaultReturnUrl": DEFAULT_RETURN_URL,
                "scopes": ["openid", "profile", "email"]
            }
        }
    }
    target_config = {
        "mcp": {
            "openApiSchema": {
                "inlinePayload": json.dumps(linkedin_openapi_spec)
            }
        }
    }
    try:
        response = gateway_client.create_gateway_target(
            name = "LinkedInAuthCode",
            description = "Target created for testing",
            credentialProviderConfigurations = [credentialProviderConfig],
            targetConfiguration= target_config,
            gatewayIdentifier=gatewayId
        )
        targetId = response["targetId"]
        print(f"Created Linkedin target {targetId} for gateway {gatewayId}")
        return targetId
    except Exception as e:
        print(e)


gateway_target_id = create_linkedin_target(gatewayId=gateway_id,
                                           provider_arn=target_cred_provider_arn)

## Processo de Session Binding da URL de Autorização OAuth2

Agora que o gateway foi criado, vamos entender o processo de session binding antes de prosseguir.

O processo de session binding da URL de autorização OAuth2 é um mecanismo de segurança crítico que garante que as sessões de autorização OAuth2 sejam devidamente associadas a usuários autenticados no AgentCore Identity. Este processo previne o sequestro de sessão e garante que os tokens OAuth sejam concedidos apenas ao usuário pretendido.

Ref: https://docs.aws.amazon.com/bedrock-agentcore/latest/devguide/oauth2-authorization-url-session-binding.html

### Como funciona o session binding
<div style="text-align:center">
    <img src="images/identity-session-binding.png" width="90%"/>
</div>

1. Invocar agente – Seu código de agente ou mcp client invoca a API GetResourceOauth2Token para recuperar uma URL de autorização, quando um usuário agente originador deseja acessar alguma aplicação ou recurso que ele/ela possui.

2. Gerar URL de autorização – O AgentCore Identity gera uma URL de autorização e URI de sessão para o usuário navegar e consentir o acesso.

3. Autorizar e obter access token – O usuário navega até a URL de autorização e concede consentimento para que seu agente acesse seu recurso. Após isso, o AgentCore Identity redireciona o navegador do usuário para o endpoint HTTPS da sua aplicação com informações contendo o usuário originador da solicitação de autorização. Neste ponto, o endpoint HTTPS da sua aplicação determina se o usuário agente originador ainda é o mesmo que o usuário atualmente logado na sua aplicação. Se eles corresponderem, o endpoint da sua aplicação invoca CompleteResourceTokenAuth para que o AgentCore Identity possa buscar e armazenar o access token.

4. Re-invocar agente para obter access token – Uma vez que a aplicação retorne uma resposta válida, sua aplicação agente poderá recuperar os access tokens OAuth2.0 que foram originalmente solicitados para o usuário. Se os usuários não corresponderem, sua aplicação simplesmente não faz nada ou registra a tentativa.

Ao permitir que o endpoint da sua aplicação verifique a identidade do usuário, o AgentCore Identity permite que sua aplicação agente garanta que é sempre o mesmo usuário que iniciou a solicitação de autorização e aquele que consentiu o acesso.

### Servidor de Callback OAuth2 Adaptável ao Ambiente

Este tutorial usa um `oauth2_callback_server.py` adaptável ao ambiente que se ajusta automaticamente a diferentes ambientes de execução:

#### **Desenvolvimento Local:**
- **URL de Callback Externa**: `http://localhost:9090/oauth2/callback` (acessível pelo navegador)
- **Comunicação Interna**: `http://localhost:9090` (notebook ↔ servidor)
- **Binding do Servidor**: `127.0.0.1` (apenas localhost, seguro)

#### **SageMaker Workshop Studio:**
- **URL de Callback Externa**: `https://<domain>.studio.<region>.sagemaker.aws/proxy/9090/oauth2/callback` (acessível pelo navegador via proxy)
- **Comunicação Interna**: `http://localhost:9090` (notebook ↔ servidor no mesmo container)
- **Binding do Servidor**: `0.0.0.0` (permite que o proxy do SageMaker alcance o servidor)

O servidor de callback OAuth2 detecta automaticamente o ambiente verificando `/opt/ml/metadata/resource-metadata.json` e se configura de acordo.

### Funcionalidades do oauth2_callback_server.py

1. **Executa um Servidor FastAPI Local** (porta 9090)
   - Fornece endpoint `/ping` para verificações de saúde
   - Fornece endpoint `/userIdentifier/token` para armazenar tokens de usuário
   - Fornece endpoint `/oauth2/callback` para lidar com redirecionamentos OAuth

2. **Gerencia o Armazenamento de Tokens de Usuário**
   - Armazena o token JWT do usuário da autenticação Cognito
   - Associa sessões OAuth com a identidade correta do usuário

3. **Processa Callbacks OAuth**
   - Recebe redirecionamentos OAuth com parâmetro `session_id`
   - Chama `CompleteResourceTokenAuth` para vincular a sessão
   - Valida a identidade do usuário antes de completar o fluxo

4. **Fornece Segurança de Sessão**
   - Garante que sessões OAuth estejam vinculadas a usuários autenticados
   - Previne acesso não autorizado a tokens OAuth

5. **Detecção de Ambiente**
   - Detecta automaticamente ambiente local vs SageMaker Studio
   - Configura URLs e binding do servidor de forma apropriada

### Considerações de Segurança

O processo de session binding OAuth2 inclui diversas medidas de segurança:
- **Validação de URL**: Apenas callback URLs pré-registradas são permitidas
- **Verificação de Sessão**: Sessões de usuário devem ser validadas antes da conclusão do token
- **Vinculação de Identidade do Usuário**: Sessões OAuth são explicitamente vinculadas a usuários autenticados
- **Isolamento de Token**: Os tokens OAuth de cada usuário são isolados e seguros
- **URLs Adaptáveis ao Ambiente**: Usa automaticamente URLs apropriadas para cada ambiente

Esta abordagem abrangente garante que os fluxos OAuth2 sejam seguros e devidamente atribuídos aos usuários corretos em ambientes multi-usuário, seja executando localmente ou no SageMaker Workshop Studio.

---

In [ ]:
# Helper function to make MCP calls
def invoke_mcp(gatewayUrl, access_token, tool_params, method = "tools/call", protocol_version = '2025-11-25'):
    headers = {
        'Content-Type': 'application/json',
        'Authorization': f'Bearer {access_token}',
        'MCP-Protocol-Version': protocol_version
    }

    payload = {
        "jsonrpc": "2.0",
        "id": 24,
        "method": method,
        "params": tool_params
    }

    try:
        response = requests.post(gatewayUrl, headers=headers, json=payload)
        request_id = response.headers.get('x-amzn-requestid') or response.headers.get('x-amz-request-id')
        print("\nAmazon Request ID:", request_id)
        response.raise_for_status()
        print(f"Invoke MCP Status Code: {response.status_code}")
        print("Response:")
        #if method != "tools/list":
        print(json.dumps(response.json(), indent=2))
        return response.json()

    except requests.exceptions.RequestException as e:
        print("Error:", e)
        if hasattr(e, 'response') and e.response is not None:
            try:
                print("Error Response:", json.dumps(e.response.json(), indent=2))
            except:
                print("Error Response Text:", e.response.text)
        raise

## Testar com MCP Client
Agora testaremos o AgentCore gateway com o MCP Client. Começaremos com o metadado **forceAuthentication = True** para garantir que possamos testar o fluxo de consentimento do usuário. Se esta é a primeira vez que você está executando esta etapa, o comportamento seria o mesmo mesmo com **forceAuthentication = False**.

Ele retornará uma Elicitation Response; que sugere fornecer o consentimento do usuário abrindo a URL.

Uma vez concluído, o servidor de callback cuidará do session binding com o AgentCore identity.

<div style="text-align:center">
    <img src="images/elicitation-resp.png" width="90%"/>
</div>

In [ ]:
import subprocess
from oauth2_callback_server import store_token_in_oauth2_callback_server, wait_for_oauth2_server_to_be_ready, get_oauth2_callback_base_url

full_scope = f"{RESOURCE_SERVER_ID}/{gateway_target_name}"
jwt_token = utils.get_token(user_pool_id=gw_user_pool_id, 
                            client_id=gw_client_id, 
                            client_secret=gw_client_secret, 
                            scope_string=full_scope, 
                            REGION=REGION)
bearer_token = jwt_token['access_token']

#Starting oAuth callback server
oauth2_callback_server_cmd = [sys.executable, "oauth2_callback_server.py", "--region", REGION]
oauth2_callback_server_process = subprocess.Popen(oauth2_callback_server_cmd)

successfully_started_oauth2_server = wait_for_oauth2_server_to_be_ready()
if not successfully_started_oauth2_server:
    print("Failed to start OAuth2 callback server to handle session binding "
          "(https://docs.aws.amazon.com/bedrock-agentcore/latest/devguide/oauth2-authorization-url-session-binding.html)")
else:
    store_token_in_oauth2_callback_server(bearer_token)

    #Test invoke tool
    CUSTOM_RETURN_URL = get_oauth2_callback_base_url() + "/oauth2/callback"
    print(f"Using callback URL: {CUSTOM_RETURN_URL}")

    _meta = {
        "aws.bedrock-agentcore.gateway/credentialProviderConfiguration": {
            "oauthCredentialProvider": {
                "returnUrl": CUSTOM_RETURN_URL,
                "forceAuthentication": True
            }
        }
    }

    print("\nTesting force re-authentication")
    resp = invoke_mcp(gatewayUrl= gateway_url, 
                      access_token=bearer_token,
                      tool_params={
                          "name": f"LinkedInAuthCode___getUserInfo",
                          "arguments": {"domainName": "integrals-dev-ed"},
                          "_meta": _meta
                        },
                      method="tools/call")

In [ ]:
#Invoking tool again after completion of oAuth flow
_meta = {
    "aws.bedrock-agentcore.gateway/credentialProviderConfiguration": {
        "oauthCredentialProvider": {
            "returnUrl": CUSTOM_RETURN_URL,
            "forceAuthentication": False
        }
    }
}
print("Invoking tool again after completion of oAuth flow")
resp = invoke_mcp(gatewayUrl= gateway_url, 
                  access_token=bearer_token,
                  tool_params={
                      "name": f"LinkedInAuthCode___getUserInfo",
                      "arguments": {"domainName": "integrals-dev-ed"},
                      "_meta": _meta
                    },
                  method="tools/call")

In [ ]:
oauth2_callback_server_process.terminate()

## Limpeza de Recursos

Execute a célula a seguir para remover todos os recursos criados durante este tutorial:

1. AgentCore Gateway e targets
2. AgentCore Identity credential provider
3. Amazon Cognito User Pool e clients
4. IAM Role

In [ ]:
# Delete AgentCore Gateway and all targets
print("Step 1: Cleaning up AgentCore Gateway resources...")
gateway_client = boto3.client('bedrock-agentcore-control', region_name=REGION)
agentcore_cleanup = utils.delete_gateway(
    gateway_client=gateway_client,
    gatewayId=gateway_id
)

# Delete AgentCore Identity Credential Provider
print("\nStep 2: Cleaning up AgentCore Identity Credential Provider...")
credential_cleanup = identity_client.delete_oauth2_credential_provider(
    name=target_cred_provider_name
)

# Delete Cognito User Pool
print("\nStep 3: Cleaning up Cognito User Pool...")
cognito_cleanup = utils.delete_cognito_user_pool(
    user_pool_id=gw_user_pool_id,
    region=REGION
)

# Delete IAM Role
print("\nStep 4: Cleaning up IAM Role...")
iam_cleanup = utils.delete_iam_role(
    role_name=f'BedrockAgentCoreGatewayRole-{timestamp}'
)

## Resumo
Você testou com sucesso o fluxo de oOutbound Auth baseado em authorization code grant recém-lançado com o AgentCore Gateway usando uma ferramenta do Linkedin como target